# 5장 1강: A/B 테스트 설계 원리와 랜덤화 — 실습문제

## 실습 목표

- A/B 테스트의 대조군, 실험군, 랜덤화 단위를 데이터에서 식별합니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 실험 목적에 맞게 사전에 정의합니다.
- 그룹 배정 수와 비율을 확인하고 계획한 50:50 배정과 일치하는지 점검합니다.
- 가설 → 설계 → 실행 → 분석의 순서로 온라인 실험계획을 작성합니다.

## 실습 환경 / 데이터

- Python, NumPy, pandas, SciPy
- `cookie_cats.csv`
- `userid`: 사용자 식별자
- `version`: 게임 게이트 위치(`gate_30`, `gate_40`)
- `sum_gamerounds`: 실험 기간 동안 플레이한 게임 라운드 수
- `retention_1`, `retention_7`: 설치 후 1일·7일 재방문 여부

> 이번 강의는 **실험 설계와 랜덤화 점검**이 중심입니다. 그룹 간 효과의 통계적 검정과 최종 배포 결정은 이후 강의에서 다룹니다.

## 실습 준비

아래 셀을 실행하여 데이터를 불러오고 크기, 결측치, 컬럼을 확인하세요.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

data_candidates = [
    Path("cookie_cats.csv"),
    Path("upload/cookie_cats.csv")]

data_path = next((path for path in data_candidates if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("cookie_cats.csv 파일을 노트북과 같은 폴더에 넣어주세요.")

df = pd.read_csv(data_path)
alpha = 0.05

print(f"데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
df.head()


데이터 크기: 90189행, 5열
전체 결측치 수: 0
컬럼: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


---

## 필수 1. Cookie Cats 실험 구조와 지표 정의

게임의 첫 번째 강제 대기 게이트를 30단계에서 40단계로 옮기면 사용자 유지율이 달라지는지 확인하려고 합니다.

### 수행 요구사항

1. `userid`의 중복 여부를 확인하여 사용자 한 명이 한 행으로 기록되었는지 점검하세요.
2. `version`의 고유값과 그룹별 사용자 수를 확인하세요.
3. 버전별 사용자 수, 평균 게임 라운드 수, 1일 유지율, 7일 유지율을 하나의 요약표로 만드세요.
4. 아래 질문에 문장으로 답하세요.

### 질문

- 이 실험의 랜덤화 단위는 무엇인가요?
- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?


In [2]:
# 여기에 코드를 작성하세요.
# ============================================
# 1. userid 중복 여부 점검
# ============================================
duplicate_users = df['userid'].duplicated().sum()
n_rows = len(df)
n_unique_users = df['userid'].nunique()

print('--- 1. userid 중복 점검 ---')
print(f"전체 행 수: {n_rows}")
print(f"고유 userid 수: {n_unique_users}")
print(f"중복된 userid 행 수: {duplicate_users}")
print()

# ============================================
# 2. version 고유값 및 그룹별 사용자 수 확인
# ============================================
versions = df['version'].unique()
group_counts = df['version'].value_counts().reindex(['gate_30', 'gate_40'])

print('--- 2. 실험 그룹(version) 확인 ---')
print(f"version 고유값: {versions}")
print("그룹별 사용자 수:")
print(group_counts)
print()

# ============================================
# 3. 버전별 요약 통계표
# ============================================
experiment_summary = (
    df.groupby('version')
    .agg(
        users=('userid', 'size'),
        avg_gamerounds=('sum_gamerounds', 'mean'),
        retention_1_rate=('retention_1', 'mean'),
        retention_7_rate=('retention_7', 'mean')
    )
    .reindex(['gate_30', 'gate_40'])
)

print('--- 3. 버전별 지표 요약표 ---')
print(experiment_summary.round(4))

--- 1. userid 중복 점검 ---
전체 행 수: 90189
고유 userid 수: 90189
중복된 userid 행 수: 0

--- 2. 실험 그룹(version) 확인 ---
version 고유값: <StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str
그룹별 사용자 수:
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64

--- 3. 버전별 지표 요약표 ---
         users  avg_gamerounds  retention_1_rate  retention_7_rate
version                                                           
gate_30  44700         52.4563            0.4482            0.1902
gate_40  45489         51.2988            0.4423            0.1820


- 이 실험의 랜덤화 단위는 무엇인가요?
<br> -> userid로 구분되는 사용자, 한 사용자는 하나의 버전에만 배정되어 일관된 게임 경험을 제공하여 비교

- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
<br> -> 기존 게이트 위치인 gate_30을 대조군, 게이트를 40단계로 옮긴 gate_40을 실험군으로 설정

- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
<br> -> 핵심 지표는 단순 1일 평가보다는 7일 이후 시점을 나타내는 retention_7로(재방문)
     -> 보조 지표는 초기 반응 확인을 위해 retention_1로
     -> 가드레일 지표는 활동이 감소하는 지 확인하기 위해 sum_gamerounds(평균 게임 라운드 수)로 보겠다.

- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?
<br> -> 아닙니다. 표본 변동으로 생길 수 있는 차이인지? 통계적으로 검정이 필요하다.

---

## 필수 2. 무작위 배정 실습과 그룹 비율 점검

실제 `version` 값은 변경하지 않고, 동일한 사용자 목록에 연습용 A/B 그룹을 새로 무작위 배정한 뒤 실제 배정 비율과 비교하세요.

### 수행 요구사항

1. `np.random.default_rng(42)`를 사용하세요.
2. 각 `userid`에 `A` 또는 `B`를 50:50 확률로 배정한 `randomized_users`를 만드세요.
3. 연습용 그룹별 인원수와 비율을 출력하세요.
4. 실제 `version`별 인원수와 비율도 출력하세요.
5. 실제 실험이 50:50 배정을 계획했다고 가정하고, 기대빈도를 전체 인원의 절반으로 설정하여 카이제곱 적합도 검정을 수행하세요.
6. 아래 질문에 답하세요.

### 질문

- 시드를 고정하는 이유는 무엇인가요?
- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?


In [5]:
# 여기에 코드를 작성하세요.
# ============================================
# 1. 난수생성기 초기화
# ============================================
rng = np.random.default_rng(42)

# ============================================
# 2. 연습용 A/B 그룹 무작위 배정
# ============================================
randomized_users = df[['userid']].copy()
randomized_users['practice_group'] = rng.choice(
    ['A', 'B'], size=len(randomized_users), p=[0.5, 0.5]
)

# ============================================
# 3. 연습용 그룹 인원수 및 비율(%)
# ============================================
practice_counts = randomized_users['practice_group'].value_counts().reindex(['A', 'B'])
practice_ratios_pct = (
    randomized_users['practice_group']
    .value_counts(normalize=True)
    .reindex(['A', 'B']) * 100
).map(lambda x: f"{x:.2f}%")

print('--- 3. 연습용(practice) 그룹 배정 결과 ---')
print("그룹별 인원수:")
print(practice_counts)
print("그룹별 비율:")
print(practice_ratios_pct)
print()

# ============================================
# 4. 실제 version별 인원수 및 비율(%)
# ============================================
actual_counts = df['version'].value_counts().reindex(['gate_30', 'gate_40'])
actual_ratios_pct = (
    df['version']
    .value_counts(normalize=True)
    .reindex(['gate_30', 'gate_40']) * 100
).map(lambda x: f"{x:.2f}%")

print('--- 4. 실제(version) 그룹 배정 결과 ---')
print("그룹별 인원수:")
print(actual_counts)
print("그룹별 비율:")
print(actual_ratios_pct)
print()

# ============================================
# 5. 카이제곱 적합도 검정 (50:50 기대 가정)
# ============================================
n_total = len(df)
expected_counts = [n_total / 2, n_total / 2]
observed_counts = actual_counts.values

chi2_stat, p_value = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

print('--- 5. 카이제곱 적합도 검정 결과 ---')
print(f"카이제곱 통계량: {chi2_stat:.4f}")
print(f"p-value: {p_value:.4f}")


--- 3. 연습용(practice) 그룹 배정 결과 ---
그룹별 인원수:
practice_group
A    44852
B    45337
Name: count, dtype: int64
그룹별 비율:
practice_group
A    49.73%
B    50.27%
Name: proportion, dtype: str

--- 4. 실제(version) 그룹 배정 결과 ---
그룹별 인원수:
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64
그룹별 비율:
version
gate_30    49.56%
gate_40    50.44%
Name: proportion, dtype: str

--- 5. 카이제곱 적합도 검정 결과 ---
카이제곱 통계량: 6.9024
p-value: 0.0086


- 시드를 고정하는 이유는 무엇인가요?
<br> -> 무작위 과정을 재현하여 수강생과 검토자가 같은 배정 결과를 얻고 코드를 확인할 수 있게 하기 위해서

- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
<br> -> userid별로 practice_group으 ㅣ고유값 수를 계산하여 최대값이 1인지 확인한다.

- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
<br> -> p-value가 0.05보다 작으므로, 50:50 계획과 통계적으로 일치한다고 보기 어렵다.

- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?
<br> -> 세 변수는 버전을 경험한 이후의 측정된 결과이므로 처치영향을 받을 수 있다.
<br> -> 균형점검에서는 실험 전에 측정된 기기, 국가 등 외부 요인 같은 공변량이 필요하지만 현 데이터에서는 제공하지 않았다.

---

## 과제. Cookie Cats A/B 테스트 전체 계획서 작성

`gate_30`을 기존 버전, `gate_40`을 새 버전으로 설정한 A/B 테스트 계획을 작성하세요.

### 수행 요구사항

1. 아래 네 단계를 모두 포함한 계획서를 작성하세요.
   - 가설 설정
   - 실험 설계
   - 실험 실행
   - 결과 분석
2. 실험 단위, 대조군, 실험군, 핵심·보조·가드레일 지표를 명시하세요.
3. 실행 전에 확인할 데이터 품질 항목을 두 가지 이상 작성하세요.
4. SUTVA 위반 또는 실험 간 간섭 가능성을 검토하세요.
5. 버전별 관측 지표를 다시 계산하되, 아직 통계적 검정을 하지 않았다는 점을 반영해 최종 의사결정을 보류하는 6~8문장의 결론을 작성하세요.

> 과제는 필수 문제와 동일한 수준입니다. 표본 크기나 MDE를 계산할 필요는 없습니다.


In [ ]:
# 여기에 코드를 작성하세요.


---

## 실습 마무리

- 어떤 문제가 있었는가?
<br> -> 그룹별 유지율과 평균 게임 라운드만 비교하여 사용자으 ㅣ고정 배정 여부, 계획한 표본 비율, 지표 사전 지정, 
<br> -> 외부 간섭, 로깅 등의 문제를 놓칠 수 있었다.

- 어떻게 개선했는가?
<br> -> 실험 단위를 userid로 명시하고 대조군, 실험군과 3가지 지표를 사전에 정의했다.
<br> -> 실제 배정 비율, 이용자 중복 체크, 50:50 적합도, 결과변수, 사전 공변량의 차이 등을 함께 점검했다.

- 무엇을 근거로 개선되었다고 판단했는가?
<br> -> 사용자 중복 0건, 실제 그룹비율 약 49.5%, 50.4%, 적합도 검정 p-value가 0.05미만을 확인하여 단순히 비슷해 보인다는 피룻 1의 판단보다는 구체적인 조사 증거를 확보했다.
<br> -> 40gate를 베포전에 기술검정을 통해 버전 업 베포 결정을 보류했다.

단순한 그룹별 지표 비교에서 놓칠 수 있는 문제와, 실험 단위·사전 지표·배정 비율·간섭 가능성을 명시하면서 설계가 어떻게 개선되었는지 정리하세요.
